# 01 · Data and exploratory analysis
**Project:** an end-to-end retail demand forecasting system. Forecasting, uncertainty, an API, CI and monitoring.
**Data:** [Rossmann Store Sales](https://www.kaggle.com/competitions/rossmann-store-sales): 1,115 stores and about 1M daily records (2013–2015).
The task is to forecast daily sales for the next **48 days**.

> Before running: accept the competition rules on Kaggle (Rules tab → *I Understand and Accept*). The Colab secret
> `KAGGLE_API_TOKEN` from Project 1 is reused. **Runtime: CPU** (about 3 min).

In [ ]:
# ---------- Setup: run this cell first (CPU runtime is enough) ----------
import os, sys, subprocess, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FC_SMOKE") == "1"
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if SMOKE:
    PROJECT_DIR = os.environ["FC_PROJECT_DIR"]
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = sorted(glob.glob("/content/drive/MyDrive/**/forecast/src/forecast/models.py", recursive=True), key=len)
    assert hits, "Upload the 'forecast' folder to your Google Drive first."
    PROJECT_DIR = str(Path(hits[0]).parents[2])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate", "fastapi", "uvicorn", "httpx"], check=True)
else:
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, f"{PROJECT_DIR}/src"); sys.path.insert(0, PROJECT_DIR)
ART, FIG = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures"
DATA = "/content/data/rossmann" if ON_COLAB else f"{PROJECT_DIR}/data"
for d in (ART, FIG, DATA): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from forecast import data as D, models as M
from forecast.features import FEATURES, build_features, fit_store_stats
if SMOKE:
    train, test, store = D.make_synthetic(25)
else:
    train, test, store = D.load(D.download(DATA))
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=str)
def load_json(name):
    return json.load(open(f"{ART}/{name}.json"))
print(f"train {train.shape} {train.Date.min().date()}..{train.Date.max().date()} | test {test.shape} | stores {store.shape[0]}")

## 1. Structure and data quality

In [ ]:
print("days per store:", train.groupby("Store").size().describe()[["min", "50%", "max"]].to_dict())
closed = (train.Open == 0).mean(); zero_open = ((train.Open == 1) & (train.Sales == 0)).sum()
print(f"closed days: {closed:.1%} | open days with zero sales: {zero_open}")
n_days = train.groupby("Store").Date.nunique()
gap_stores = n_days[n_days < n_days.max()].index
print(f"{len(gap_stores)} stores have missing dates (Rossmann's known mid-2014 gap for refurbished stores)")
store.isna().mean().round(3).to_frame("missing share")

In [ ]:
op = train[train.Open == 1]
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
op.groupby("Date").Sales.mean().rolling(7).mean().plot(ax=ax[0, 0]); ax[0, 0].set_title("Mean daily sales per open store (7-day avg)")
op.groupby("DayOfWeek").Sales.mean().plot.bar(ax=ax[0, 1], color="#4C72B0"); ax[0, 1].set_title("By day of week")
op.groupby(["Promo"]).Sales.mean().plot.bar(ax=ax[1, 0], color=["#BBBBBB", "#C44E52"]); ax[1, 0].set_title("Promo vs no promo")
op.merge(store[["Store", "StoreType"]]).groupby("StoreType").Sales.mean().plot.bar(ax=ax[1, 1], color="#55A868")
ax[1, 1].set_title("By store type"); savefig("01_patterns")

In [ ]:
lift = op.groupby("Promo").Sales.mean(); print(f"promo days: +{lift[1]/lift[0]-1:.1%} sales on average")
per_store = op.groupby("Store").Sales.mean()
print(f"store mean sales: min {per_store.min():.0f}, median {per_store.median():.0f}, max {per_store.max():.0f}  -> "
      f"{per_store.max()/per_store.min():.0f}x spread: a global model needs store-level information")
dec = op[op.Date.dt.month == 12].groupby(op.Date.dt.day).Sales.mean()
dec.plot(figsize=(7, 3), marker="o", title="December: mean sales by day of month"); savefig("01_december")

In [ ]:
save_json({"n_rows": len(train), "n_stores": int(store.shape[0]), "closed_share": float(closed),
           "promo_lift": float(lift[1] / lift[0] - 1), "store_spread": float(per_store.max() / per_store.min()),
           "n_gap_stores": int(len(gap_stores))}, "eda")
print("saved")

## Observations (fill in yourself)
* Which patterns (weekday, promo, seasonality, store type) must the model capture?
* Data-quality issues and how they're handled.